In [1]:
# pyright: ignore
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from import_ipynb import NotebookFinder
import importlib
import joblib
import os

## Configuration et chargement des modules

Configuration des chemins de répertoires et chargement dynamique des modules utilitaires:
- Détermination du répertoire racine du projet
- Accès au répertoire contenant les hyperparamètres sauvegardés
- Chargement du module **metrics** pour les métriques d'évaluation personnalisées
- Chargement du module **prints** pour les fonctions d'affichage formatées

In [2]:
# retrouver les dossiers
root = r"C:\Travaux\Epitech\Zoidberg2.0"
hyperparameter_dir = os.path.join(root, "pneumonia_knn","document","result", "model", "hyperparameter")
utility_dir = os.path.join(root, "pneumonia_knn", "utility")
print(os.listdir(utility_dir))
# charger les fichiers
# --- pneumonia_knn\notebooks\visualisation\metrics.ipynb
spec_metrics = NotebookFinder().find_spec("metrics", [utility_dir])
metrics = importlib.util.module_from_spec(spec_metrics)
spec_metrics.loader.exec_module(metrics)

# --- pneumonia_knn\notebooks\utils
spec_print = NotebookFinder().find_spec("prints", [utility_dir])
prints = importlib.util.module_from_spec(spec_print)
spec_print.loader.exec_module(prints)

['dataset.ipynb', 'metrics.ipynb', 'prints.ipynb']


## Création du pipeline PCA + KNN

Création d'un pipeline scikit-learn combinant la réduction de dimensionnalité (PCA) et la classification (KNN).

**Pipeline:**
- Étape 1: PCA pour réduire la dimensionnalité
- Étape 2: KNeighborsClassifier pour la classification

**Hyperparamètres à tester:**
- **PCA n_components**: 0.95 (conserve 95% de la variance)
- **KNN n_neighbors**: [3, 5, 9] (nombre de voisins)
- **KNN metric**: ['euclidean', 'manhattan'] (distance utilisée)
- **KNN weights**: ['distance'] (pondération par la distance)
- **KNN algorithm**: ['ball_tree'] (structure de données pour la recherche)

In [3]:
# 1. Créer le pipeline
pipeline = Pipeline([
    ('pca', PCA()),
    ('knn', KNeighborsClassifier())
])

# 2. Définir les hyperparamètres
param_grid = {
    'pca__n_components': [0.95],
    'knn__n_neighbors': [3, 5, 9],
    'knn__metric': ['euclidean', 'manhattan'],
    'knn__weights': ['distance'],
    'knn__algorithm': ['ball_tree']
}

## GridSearchCV pour optimiser les hyperparamètres

Exécution d'une recherche exhaustive des hyperparamètres optimaux via GridSearchCV:
- Utilisation d'une **validation croisée** avec cv=2
- Optimisation basée sur plusieurs **métriques d'évaluation** (via get_scoring())
- **refit='recall'**: le meilleur modèle est sélectionné selon la métrique recall
- Le modèle est **sauvegardé** en cache pour éviter les recalculs
- Si un modèle optimisé existe déjà, il est directement chargé

In [4]:
file_name = "knn_pca_grid_search.pkl"
# Sauvegarder le résultat de GridSearchCV pour éviter de le relancer à chaque fois, s'il n'existe pas déjà.
if not os.path.exists(f'{hyperparameter_dir}/{file_name}'):
    prints.bold("Process : Cross Validation - PCA + KNN")
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=2,
        scoring=metrics.get_scoring(),
        refit='recall',
        n_jobs=1,
        verbose=2
    )
    grid.fit(X_train, y_train) # type: ignore
    joblib.dump(grid, f'{hyperparameter_dir}/{file_name}')

grid = joblib.load(f'{hyperparameter_dir}/{file_name}')

## Transformations PCA

Application de la transformation PCA optimisée sur les données d'entraînement et de test:
- Utilisation du modèle PCA du meilleur estimateur trouvé par GridSearchCV
- Transformation de **X_train** en **X_train_pca** (données réduites d'entraînement)
- Transformation de **X_test** en **X_test_pca** (données réduites de test)
- Les données sont réduites à 95% de la variance originale

In [5]:
# Créer les nouvelles données transformées par PCA pour les utiliser dans les autres algorithmes
X_train_pca = grid.best_estimator_.named_steps['pca'].transform(X_train) # type: ignore
X_test_pca = grid.best_estimator_.named_steps['pca'].transform(X_test) # type: ignore

NameError: name 'X_train' is not defined

## Prédictions

Génération des prédictions sur l'ensemble de test avec le meilleur modèle:
- Extraction du meilleur estimateur (pipeline PCA + KNN) trouvé par GridSearchCV
- Prédiction des labels pour les données de test (**y_pred_pca**)
- Ces prédictions seront comparées avec les labels réels pour évaluer les performances

In [ ]:
knn_pca = grid.best_estimator_
y_pred_pca = knn_pca.predict(X_test) # type: ignore

## Affichage des résultats

GridSearchCV teste automatiquement toutes les combinaisons de paramètres (fine tuning / hyperparameter tuning) que vous lui fournissez, et pour chaque combinaison il effectue une validation croisée (cross-validation) afin d'évaluer la performance de façon fiable. 

Ici les résultats sont affichés afin de voir quelle combinaison de paramètres a obtenu le meilleur score moyen.

Si vous avez lancez la pipeline via le main.ipynb vous retrouverez les résultats dans les fichiers suivants : 
    - [knn_lda.ipynb](../results/process/knn_lda.ipynb)
grid_search_metrics_table_PCA

In [ ]:
# Afficher les résultats de la grille de recherche - PCA + KNN - Cross Validation
metrics.cross_validation_png(grid, "PCA")
# Enregistrer le tableau de résultats de la grille de recherche sous forme json - PCA + KNN - Cross Validation
metrics.cross_validation_json(grid, "PCA")